# Faza 0: Nalaganje podatkov in vzorčenje

**Zastavljen cilj**: Pripravili obvladljivo in čisto začetno bazo iz izhodiščne ~1,2 GB datoteke `accepted_2007_to_2018Q4.csv`. Obdržali smo ločena popolnoma poplačana in neplačana posojila, izbrali bistvene atribute ter kreirali stratificiran vzorec 20.000 vrstic za hitrejše nadaljnje modeliranje.

## 1. Uvoz knjižnic

Najprej smo uvozili vse osnovne knjižnice, ki so bile potrebne za operacije nad podatkovnimi okviri in za razdelitev podatkov.

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# TODO (ekipa): Uvozi še druge knjižnice, ki so / bodo prišle prav v tej fazi.

## 2. Nalaganje podatkov

Zaradi velikosti izvorne datoteke smo zagotovili uporabo knjižnice `pandas` s parametrom `low_memory=False`. Ta parameter nam je omogočil varno prebiranje in preprečil prekinitve ob morebitnih opozorilih o mešanih podatkovnih tipih. Začasno smo tudi testirali nalaganje prvih nekaj tisoč vrstic preden smo pospešeno naložili celoto.

In [19]:
# ✔ TODO (ekipa): Preberi CSV datoteko `../data/raw/accepted_2007_to_2018Q4.csv` v nov podatkovni okvir z imenom 'df'
# ✔ TODO (ekipa): Prikaži osnovno obliko (npr. z `df.shape`) in izpiši prvih nekaj vrstic.

file_path = '../data/raw/accepted_2007_to_2018Q4.csv'

df = pd.read_csv(file_path, low_memory=False)

#print([col for col in df.columns if 'percent' in col.lower()])

print(f"Oblika podatkovnega okvirja: {df.shape}")
df.head()

Oblika podatkovnega okvirja: (2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Filtriranje in kodiranje ciljne spremenljivke (`loan_status`)

Ker v naši raziskavi obravnavamo binarni klasifikacijski problem napovedovanja neplačila, smo preiskali obsežen originalni seznam statusov:

1. **Obdržali** smo le tiste vrstice, pri katerih je bil proces posojila dokončno in uradno zaključen (status `'Fully Paid'` ali `'Charged Off'`).
2. **Kodirali** smo poenostavljeno ciljno spremenljivko na način, da je uspešen konec `'Fully Paid'` predstavljen z vrednostjo `0`, ter neuspeh `'Charged Off'` z vrednostjo `1`. Vse nadaljnje analize (zlasti XGboost in SHAP) to sedaj tolmačijo kot uresničenje nezaželenega dogodka (ang. *default*).

In [20]:
# ✔ TODO (ekipa): S pomočjo df['loan_status'].isin(...) obdrži samo vrstice za naštete dokončane kredite.
# ✔ TODO (ekipa): Preslikaj besedilne statuse (npr. z .map({'Fully Paid':0, 'Charged Off':1})) v navadne inte (0 in 1).
# ✔ TODO (ekipa): Prikaži, da je bilo uspešno, in preveri vsebino z `df['loan_status'].value_counts()`.

df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]
df['loan_status'] = df['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})

print(df['loan_status'].value_counts())

loan_status
0    1076751
1     268559
Name: count, dtype: int64


## 4. Izbira spremenljivk

Sam izvirni nabor zajema več kot 150 stolpcev. Ker številni navajajo podatke, ki nam na točkovni začetni odobritvi posojila ne bi bili javno na voljo (t.i. uhajanje podatkov iz prihodnosti ali *Data Leakage*), ali pa so na spisku delovali popolnoma nerelevantno oziroma prazno, smo zadržali zgolj specifično in vnaprej izbrano podmnožico.

**Ohranili smo naslednje specifične stolpce:**
`loan_amnt`, `funded_amnt`, `term`, `int_rate`, `installment`, `grade`, `sub_grade`, `emp_title`, `emp_length`, `home_ownership`, `annual_inc`, `verification_status`, `desc`, `purpose`, `dti`, `delinq_2yrs`, `earliest_cr_line`, `fico_range_low`, `fico_range_high`, `inq_last_6mths`, `mths_since_last_delinq`, `open_acc`, `pub_rec`, `revol_bal`, `revol_util`, `total_acc`, `application_type`, `tot_cur_bal`, `total_rev_hi_lim`, `acc_open_past_24mths`, `avg_cur_bal`, `bc_open_to_buy`, `bc_util`, `chargeoff_within_12_mths`, `delinq_amnt`, `mo_sin_old_il_acct`, `mo_sin_old_rev_tl_op`, `mo_sin_rcnt_rev_tl_op`, `mo_sin_rcnt_tl`, `mort_acc`, `num_accts_ever_120_pd`, `percent_bc_gt75`, `pub_rec_bankruptcies`, `tax_liens`, `loan_status`

*(Opis/opozorilo: Stolpec za zvezno državo (`addr_state`) je v skladu z dogovorom namerno izključen, saj preveč poslabša the dimenzionalnost naše enačbe s kar 50 novimi stolpci po kodiranju, medtem ko obenem prinaša le skromno informacijsko vrednost.)*


In [21]:
# ✔ TODO (ekipa): Predaj dogovorjeni seznam obdržanih imen (iz zgoraj) v obliki string liste.
# ✔ TODO (ekipa): Prepiši in zmanjšaj trenutno tabelo: npr. df = df[columns_to_keep].
# ✔ TODO (ekipa): Poglej prečiščene stolpce z `df.info()` ali z izpisom `.columns`.

columns_to_keep = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 
    'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 
    'desc', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 
    'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 
    'revol_bal', 'revol_util', 'total_acc', 'application_type', 'tot_cur_bal', 
    'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 
    'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 
    'num_accts_ever_120_pd', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'loan_status'
]

df = df[columns_to_keep]
df.info()

<class 'pandas.DataFrame'>
Index: 1345310 entries, 0 to 2260697
Data columns (total 45 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   loan_amnt                 1345310 non-null  float64
 1   funded_amnt               1345310 non-null  float64
 2   term                      1345310 non-null  str    
 3   int_rate                  1345310 non-null  float64
 4   installment               1345310 non-null  float64
 5   grade                     1345310 non-null  str    
 6   sub_grade                 1345310 non-null  str    
 7   emp_title                 1259525 non-null  str    
 8   emp_length                1266799 non-null  str    
 9   home_ownership            1345310 non-null  str    
 10  annual_inc                1345310 non-null  float64
 11  verification_status       1345310 non-null  str    
 12  desc                      123530 non-null   str    
 13  purpose                   1345310 non-null 

## 5. Stratificirano vzorčenje na n=20.000

Zaradi smotrno skrajšanih časov učenja in lažjega reševanja NLP problema ob koncu (TF-IDF zamenjava na opisih), smo obširno polno množico skrčili točno na predstavitveni podvzorec obsega 20.000 inštanc.

Pozornost pri tem se je najbolj namenila **stratifikaciji**, in sicer z uporabo parametra ali ustreznega grupiranja nad `loan_status`. Tako smo zmanjšali podatke na številko 20.000, pri čemer je sedaj delež problematičnih (1) ter normalnih (0) takoj udeležen identično kakor pri polnih vseh ~1,4 miljonih podatkih.

In [22]:
# ✔ TODO (ekipa): Pripravi in uporabi algoritem podvzorčenja s stratifikacijo. (Npr. groupby in frac ali train_test_split s fiksnim train_size=20000/len(df) ). Določi seme random_state=42.
# ✔ TODO (ekipa): Razmerje razredov ustrezno preveri z izpisom (npr. s `normalize=True`).
# ✔ TODO (ekipa): Ne pozabite shraniti končen ekstrahiran podvzorec `../data/lending_club_20k.csv` (npr. z df.to_csv in index=False).

sample_size = 20000
fraction = sample_size / len(df)

df_sample, _ = train_test_split(
    df, 
    train_size=fraction, 
    stratify=df['loan_status'], 
    random_state=42
)

print("Razmerje razredov v vzorcu:")
print(df_sample['loan_status'].value_counts(normalize=True))

df_sample.to_csv('../data/lending_club_20k.csv', index=False)
print(f"Vzorec shranjen. Nova oblika: {df_sample.shape}")

Razmerje razredov v vzorcu:
loan_status
0    0.80035
1    0.19965
Name: proportion, dtype: float64
Vzorec shranjen. Nova oblika: (20000, 45)


## 6. Delitev na učno in testno množico

Tako dobljen in prečiščen, a sedaj zmanjšan stratificirani nabor (20.000 instanc), je na konec bil strogo razčlenjen v izbranem razmerju **80 proti 20 (80 : 20)** med dve popolnoma neodvisni množici: *Train* in *Test*.

Tudi pri tem ultimatem pomembnem deju se je upoštevalo vzdrževanje omenjenih enakovrebih porazdelitev (ponovna stratifikacija nad statusi posojila).

In [23]:
# ✔ TODO (ekipa): Izvedi razdelitev prejšnjih 20k na 80/20. Z uporabo `train_test_split(df_sample, test_size=0.2, stratify=df_sample['loan_status'], random_state=42)`.
# ✔ TODO (ekipa): Nauči in shrani dobljeno ločeno učno podmnožico v datoteki `../data/train_raw.csv` z index=False.
# ✔ TODO (ekipa): Izvozi testno podmnožico v posamezno `../data/test_raw.csv` (index=False).
# ✔ TODO (ekipa): Natisni velikosti teh polj, da razjasni dimenzijo na samem koncu Notebooka.

df_train, df_test = train_test_split(
    df_sample, 
    test_size=0.20, 
    stratify=df_sample['loan_status'], 
    random_state=42
)

df_train.to_csv('../data/train_raw.csv', index=False)
df_test.to_csv('../data/test_raw.csv', index=False)

print("--- Končne dimenzije ---")
print(f"Učna množica (Train): {df_train.shape}")
print(f"Testna množica (Test): {df_test.shape}")

print("\nPorazdelitev v testni množici:")
print(df_test['loan_status'].value_counts(normalize=True))

--- Končne dimenzije ---
Učna množica (Train): (16000, 45)
Testna množica (Test): (4000, 45)

Porazdelitev v testni množici:
loan_status
0    0.80025
1    0.19975
Name: proportion, dtype: float64
